# Gemma Scope 2: Gemma 3 270M resid-post

Notebook Kaggle uruchamia wyłącznie analizę gotowego SAE Gemma Scope 2. Nie zawiera treningu SAE. Model `google/gemma-3-270m` jest analizowany na tym samym pliku The Pile co pipeline Pythii, a wyniki trafiają do `/kaggle/working/gemma_scope2_270m_pilecc/analysis/`.

Domyślnie używana jest środkowa warstwa `9` i `layer_9_width_16k_l0_medium` z release SAELens `gemma-scope-2-270m-pt-res`, który wskazuje katalog `resid_post` w repozytorium Gemma Scope 2.

In [ ]:
import os
import numpy as np
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# Kaggle zwykle ma starszy Transformers; Gemma 3 wymaga nowszej wersji.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers==4.57.6',
    'sae-lens==6.50.0',
    'transformer-lens==3.2.1',
], check=True)
import sae_lens
print('Dependencies ready; sae-lens', getattr(sae_lens, '__version__', '<unknown>'))

## Model bazowy z Kaggle Models

Model `google/gemma-3-270m` jest podpinany przez pipeline jako `google/gemma-3/transformers/gemma-3-270m/2`. Każde konto uruchamiające notebook musi wcześniej zaakceptować warunki Gemmy na Kaggle. Model jest ładowany z lokalnego, tylko do odczytu katalogu `/kaggle/input`, więc notebook nie wymaga sekretu `HF_TOKEN`.

In [ ]:
KAGGLE_MODEL_SOURCE = 'google/gemma-3/transformers/gemma-3-270m/2'

def resolve_kaggle_model_dir(owner, model_slug, framework, variation, version):
    candidates = [
        Path('/kaggle/input/models') / owner / model_slug / framework / variation / version,
        Path('/kaggle/input') / model_slug / framework / variation / version,
    ]
    for path in candidates:
        if path.is_dir():
            if not (path / 'config.json').is_file():
                raise FileNotFoundError(
                    f'Podpięty Kaggle Model nie zawiera config.json: {path}'
                )
            return path
    raise FileNotFoundError(
        f'Model {KAGGLE_MODEL_SOURCE} nie jest podpięty; sprawdzono: {candidates}'
    )

GEMMA_MODEL_DIR = resolve_kaggle_model_dir(
    'google', 'gemma-3', 'transformers', 'gemma-3-270m', '2'
)
print('Kaggle Model ready:', GEMMA_MODEL_DIR)

In [ ]:
# =========================
# JEDYNE USTAWIENIA SESJI
# =========================
VERBOSE = 'low'
VERBOSE_INTERVAL = 1000
SEQUENCE_VERBOSE_INTERVAL = 1_000_000

# None oznacza analizę wszystkich przygotowanych sekwencji.
ANALYSIS_MAX_SEQUENCES = None
ANALYSIS_SAMPLING = 'uniform'
RESUME_ANALYSIS = True
CHECKPOINT_EVERY_BATCHES = 25
SEQUENCE_MAX_TOKENS = None  # np. 5_000_000 dla pilota testowego
SEQUENCE_MAX_DOCUMENTS = None

# Jeśli True, użyj gotowych plików z Kaggle Input i pomiń sekwencjonowanie The Pile.
USE_PREPARED_SEQUENCES = True
REQUIRE_CHECKPOINT_INPUT = False  # False pozwala uruchomić pierwszy run Gemmy od zera.

def resolve_kaggle_dataset_dir(owner, slug):
    candidates = [
        Path('/kaggle/input/datasets') / owner / slug,
        Path('/kaggle/input') / slug,
    ]
    matches = [path for path in candidates if path.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f'Dataset {owner}/{slug} nie jest podpięty; sprawdzono: {candidates}'
        )
    return matches[0]

PREPARED_SEQUENCE_DIR = resolve_kaggle_dataset_dir('erykmikoajek', 'trained-sae-models')
PREPARED_TOKENS_PATH = PREPARED_SEQUENCE_DIR / 'tokens_seqs_padded_gemma.npy'
PREPARED_MASK_PATH = PREPARED_SEQUENCE_DIR / 'attention_mask_gemma.npy'

CODE_DIR = resolve_kaggle_dataset_dir('erykmikoajek', 'sae-training-and-moeffication')
os.environ['PYTHONPATH'] = str(CODE_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
ANALYZER = CODE_DIR / 'sae_pipeline' / 'gemma_scope_analysis.py'
RESULTS_DIR = Path('/kaggle/working/gemma_scope2_270m_pilecc')
# Opcjonalnie: checkpoint zapisany wcześniej jako plik Kaggle Dataset.
# Po skopiowaniu do working analizator będzie go aktualizował lokalnie.
CHECKPOINT_INPUT_PATH = PREPARED_SEQUENCE_DIR / 'analysis_checkpoint_gemma_scope2_270m.pt'
CHECKPOINT_WORKING_PATH = RESULTS_DIR / 'analysis' / 'analysis_checkpoint_gemma_scope2_270m.pt'
# Ta sama ścieżka do źródła The Pile jak w kaggle_pythia160m_sae.ipynb.
PILE_DATASET_DIR = None
if not USE_PREPARED_SEQUENCES:
    PILE_DATASET_DIR = resolve_kaggle_dataset_dir('dschettler8845', 'the-pile-dataset-part-00-of-29')
PILE_JSONL = PILE_DATASET_DIR / '00.jsonl' if PILE_DATASET_DIR is not None else None

MODEL_NAME = str(GEMMA_MODEL_DIR)
TOKENIZER_NAME = MODEL_NAME
SAE_RELEASE = 'gemma-scope-2-270m-pt-res'
SAE_ID = 'layer_9_width_16k_l0_medium'
LAYER_NUM = 9
SEQ_LENGTH = 1024
MODEL_BATCH_SIZE = 2
CHUNK_SEQUENCES = 8
SAE_BATCH_SIZE = 256
TOP_K_EXAMPLES = 15
TOP_K_TOKENS = 10
CONTEXT_SIZE = 25
COMPUTE_LOGIT_LENS = True
LOGIT_TOP_K = 128
LOGIT_BLOCK_SIZE = 64

if VERBOSE not in {'low', 'high'}:
    raise ValueError('VERBOSE musi być low albo high')
if ANALYSIS_SAMPLING not in {'head', 'tail', 'uniform'}:
    raise ValueError('ANALYSIS_SAMPLING musi być head, tail albo uniform')
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f'Brak katalogu ze źródłami: {CODE_DIR}')
if not ANALYZER.is_file():
    raise FileNotFoundError(f'Brak analizatora: {ANALYZER}')
if PILE_JSONL is not None and not PILE_JSONL.is_file():
    raise FileNotFoundError(f'Brak źródła The Pile: {PILE_JSONL}')

print('Model:', MODEL_NAME)
print('SAE:', SAE_RELEASE, SAE_ID)
print('Layer/site: resid_post', LAYER_NUM)
print('Raw Pile shard (fallback only):', PILE_JSONL or '<not mounted>')
print('Results:', RESULTS_DIR)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>'))

In [ ]:
def run_analysis(*extra_args):
    command = [
        sys.executable, str(ANALYZER),
        '--model-name', MODEL_NAME,
        '--tokenizer-name', TOKENIZER_NAME,
        '--release', SAE_RELEASE,
        '--sae-id', SAE_ID,
        '--data-path', str(RESULTS_DIR),
        '--seq-length', str(SEQ_LENGTH),
        '--layer-num', str(LAYER_NUM),
        '--model-batch-size', str(MODEL_BATCH_SIZE),
        '--chunk-sequences', str(CHUNK_SEQUENCES),
        '--sae-batch-size', str(SAE_BATCH_SIZE),
        '--top-k-examples', str(TOP_K_EXAMPLES),
        '--top-k-tokens', str(TOP_K_TOKENS),
        '--context-size', str(CONTEXT_SIZE),
        '--logit-top-k', str(LOGIT_TOP_K),
        '--logit-block-size', str(LOGIT_BLOCK_SIZE),
        '--verbose', VERBOSE,
        '--verbose-interval', str(VERBOSE_INTERVAL),
        '--checkpoint-every-batches', str(CHECKPOINT_EVERY_BATCHES),
    ]
    if PILE_JSONL is not None:
        command.extend(['--input-path', str(PILE_JSONL)])
    if CHECKPOINT_INPUT_PATH.is_file():
        if not CHECKPOINT_WORKING_PATH.is_file():
            CHECKPOINT_WORKING_PATH.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(CHECKPOINT_INPUT_PATH, CHECKPOINT_WORKING_PATH)
            print(f'Przywrócono checkpoint do working: {CHECKPOINT_WORKING_PATH}')
    elif REQUIRE_CHECKPOINT_INPUT:
        raise FileNotFoundError(f'Brak wymaganego checkpointu wejściowego: {CHECKPOINT_INPUT_PATH}')
    else:
        print('Brak checkpointu wejściowego: analiza Gemmy rozpocznie się od początku.')
    command.extend(['--checkpoint-path', str(CHECKPOINT_WORKING_PATH)])
    if USE_PREPARED_SEQUENCES:
        command.extend([
            '--tokens-path', str(PREPARED_TOKENS_PATH),
            '--attention-mask-path', str(PREPARED_MASK_PATH),
            '--require-prepared-sequences',
        ])
    if not COMPUTE_LOGIT_LENS:
        command.append('--skip-logit-lens')
    if not RESUME_ANALYSIS:
        command.append('--no-resume')
    if ANALYSIS_MAX_SEQUENCES is not None:
        command.extend(['--max-sequences', str(ANALYSIS_MAX_SEQUENCES), '--sampling', ANALYSIS_SAMPLING])
    if SEQUENCE_MAX_TOKENS is not None:
        command.extend(['--max-tokens', str(SEQUENCE_MAX_TOKENS)])
    if SEQUENCE_MAX_DOCUMENTS is not None:
        command.extend(['--max-documents', str(SEQUENCE_MAX_DOCUMENTS)])
    command.extend(map(str, extra_args))
    print(shlex.join(command))
    subprocess.run(command, check=True)

# Preflight przed długim przebiegiem.
help_probe = subprocess.run([sys.executable, str(ANALYZER), '--help'], capture_output=True, text=True)
if help_probe.returncode != 0:
    raise RuntimeError(help_probe.stdout + help_probe.stderr)
required_flags = ['--sae-id', '--skip-logit-lens', '--tokens-path', '--attention-mask-path', '--require-prepared-sequences', '--checkpoint-every-batches', '--no-resume']
if any(flag not in help_probe.stdout for flag in required_flags):
    raise RuntimeError('Nieaktualny gemma_scope_analysis.py w Kaggle dataset.')
analyzer_source = ANALYZER.read_text(encoding='utf-8')
required_markers = ['_checkpoint_config_matches', 'sampling_layout', 'is_bf16_supported']
missing_markers = [marker for marker in required_markers if marker not in analyzer_source]
if missing_markers:
    raise RuntimeError(
        f'Nieaktualny gemma_scope_analysis.py; brak elementów: {missing_markers}. '
        'Ponownie opublikuj dataset erykmikoajek/sae-training-and-moeffication.'
    )
print('Analyzer preflight OK')

In [ ]:
# Zweryfikuj pliki wejściowe bez kopiowania ich do /kaggle/working.
if USE_PREPARED_SEQUENCES:
    if not PREPARED_TOKENS_PATH.is_file():
        raise FileNotFoundError(
            f'Brak gotowych sekwencji: {PREPARED_TOKENS_PATH}'
        )
    if not PREPARED_MASK_PATH.is_file():
        raise FileNotFoundError(
            f'Brak gotowej maski atencji: {PREPARED_MASK_PATH}'
        )
    token_shape = np.load(PREPARED_TOKENS_PATH, mmap_mode='r').shape
    mask_shape = np.load(PREPARED_MASK_PATH, mmap_mode='r').shape
    if len(token_shape) != 2 or token_shape != mask_shape or token_shape[1] != SEQ_LENGTH:
        raise ValueError(
            f'Niezgodne przygotowane sekwencje: tokens={token_shape}, mask={mask_shape}, '
            f'oczekiwana długość={SEQ_LENGTH}'
        )
    print('Gotowe sekwencje i maska atencji będą przekazane bezpośrednio do analizatora.')
else:
    if PILE_JSONL is None or not PILE_JSONL.is_file():
        raise FileNotFoundError(f'Brak źródła The Pile: {PILE_JSONL}')
    print('USE_PREPARED_SEQUENCES=False: analizator przygotuje sekwencje z The Pile, jeśli ich brakuje.')

print('Tokens:', PREPARED_TOKENS_PATH)
print('Attention mask:', PREPARED_MASK_PATH)

## Analiza Gemma Scope 2

Domyślnie notebook używa gotowych `tokens_seqs_padded_gemma.npy` i `attention_mask_gemma.npy` z datasetu `trained-sae-models`, przekazując je bezpośrednio do analizatora. Dzięki temu nie wykonuje ponownie sekwencjonowania The Pile ani nie kopiuje plików do `/kaggle/working`. Ustaw `USE_PREPARED_SEQUENCES = False`, jeśli chcesz przygotować sekwencje z `PILE_JSONL`. Następnie model Gemma 3 270M zostanie przepuszczony przez dane, a aktywacje `resid_post` będą zakodowane przez gotowy SAE Gemma Scope 2.

In [ ]:
run_analysis()

In [ ]:
print('Gotowe.')
print('Feature cards:', RESULTS_DIR / 'analysis' / 'feature_cards.jsonl')
print('JSON analysis:', RESULTS_DIR / 'analysis' / 'feature_analysis.json')
print('Text analysis:', RESULTS_DIR / 'analysis' / 'feature_analysis.txt')
if not CHECKPOINT_WORKING_PATH.is_file():
    raise RuntimeError(f'Analiza nie zapisała checkpointu: {CHECKPOINT_WORKING_PATH}')
print('Checkpoint output:', CHECKPOINT_WORKING_PATH, f'({CHECKPOINT_WORKING_PATH.stat().st_size:,} B)')